In [8]:
# basic imports
import pandas as pd
import sys
import numpy as np
import pandas as pd
import scanpy as sc

# add `Tangram` to path
import tangram as tg

sc_file_path = "/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad"
spatial_file_path = "/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad"
celltype_key = "cell_type"
output_file_path = "./deconv_results/STARmap"

ad_sc = sc.read_h5ad(sc_file_path)
ad_sp = sc.read_h5ad(spatial_file_path)

# use raw count both of scrna and spatial
sc.pp.normalize_total(ad_sc)
celltype_counts = ad_sc.obs[celltype_key].value_counts()
celltype_drop = celltype_counts.index[celltype_counts < 2]
print(f'Drop celltype {list(celltype_drop)} contain less 2 sample')
ad_sc = ad_sc[~ad_sc.obs[celltype_key].isin(celltype_drop),].copy()
sc.tl.rank_genes_groups(ad_sc, groupby=celltype_key, use_raw=False)
markers_df = pd.DataFrame(ad_sc.uns["rank_genes_groups"]["names"]).iloc[0:200, :]
print(markers_df)
genes_sc = np.unique(markers_df.melt().value.values)
print(genes_sc)
genes_st = ad_sp.var_names.values
genes = list(set(genes_sc).intersection(set(genes_st)))

tg.pp_adatas(ad_sc, ad_sp, genes=genes)

ad_map = tg.map_cells_to_space(
                   ad_sc,
                   ad_sp,
                   mode='clusters',
                   cluster_label=celltype_key)

tg.project_cell_annotations(ad_map, ad_sp, annotation=celltype_key)

# 从映射矩阵提取细胞类型组成
# ad_map.X 是 [n_clusters, n_spots] 的矩阵，表示每个簇在每个 spot 的概率/比例
# ad_map.obs 中直接包含 cell_type 列

# 直接使用 ad_map.obs 中的 cell_type
celltype_names = ad_map.obs['cell_type'].tolist()
print(f"Cell types: {celltype_names}")

# 转置以获得 [n_spots, n_celltypes] 的组成矩阵
cell_composition_tangram = pd.DataFrame(
    ad_map.X.T,  # 转置
    index=ad_sp.obs_names,  # spot 名称
    columns=celltype_names  # celltype 名称
)

print(f"\nCell composition matrix shape: {cell_composition_tangram.shape}")
print(f"Column names: {cell_composition_tangram.columns.tolist()}")
print(f"\nFirst few rows:\n{cell_composition_tangram.head()}")

# 验证每行和（应该接近 1）
row_sums = cell_composition_tangram.sum(axis=1)
print(f"\nRow sums - min: {row_sums.min():.4f}, max: {row_sums.max():.4f}, mean: {row_sums.mean():.4f}")

# 保存结果
# 🔧 归一化 Tangram 结果
print("\n" + "="*60)
print("NORMALIZING TANGRAM RESULTS")
print("="*60)

# 检查当前的行和
row_sums = cell_composition_tangram.sum(axis=1)
print(f"\n❌ BEFORE normalization:")
print(f"   Row sums - min: {row_sums.min():.6f}, max: {row_sums.max():.6f}, mean: {row_sums.mean():.6f}")
print(f"   Example: first row sum = {row_sums.iloc[0]:.6f}")

# 归一化：每行除以该行的和
cell_composition_tangram = cell_composition_tangram.div(row_sums, axis=0)

# 验证归一化后
row_sums_after = cell_composition_tangram.sum(axis=1)
print(f"\n✅ AFTER normalization:")
print(f"   Row sums - min: {row_sums_after.min():.6f}, max: {row_sums_after.max():.6f}, mean: {row_sums_after.mean():.6f}")
print(f"   Example: first row sum = {row_sums_after.iloc[0]:.6f}")

print(f"\nFirst row values (now as percentages):")
print(cell_composition_tangram.head())

# 重新保存归一化后的结果
cell_composition_tangram.to_csv("STARmap_tangram_composition.csv")
print(f"\n💾 Saved normalized results!")
print("="*60)


normalizing counts per cell
    finished (0:00:00)
Drop celltype [] contain less 2 sample
ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:33)
        Astro     Endo Excitatory L2/3   Excitatory L4 Excitatory L5  \
0      Malat1      Bsg          Ppp3ca           Nrxn1         Mgst3   
1      Slc1a3    Cldn5            Chn1            Rgs4          Chn1   
2       Ntsr2     Ly6e          Fkbp1a           Stx1a          Pcp4   
3       Aldoc    Ly6c1          Cacng3           Mef2c         Stmn1   
4      Slc1a2     Pltp           Ctxn1          Ppp3ca          Pak1   
..        ...      ...             ...             ...           ...   
195       Ubc  Tsc22d3

INFO:root:459 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:882 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 459 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.564, KL reg: 0.216
Score: 0.797, KL reg: 0.001
Score: 0.797, KL reg: 0.001
Score: 0.797, KL reg: 0.001
Score: 0.797, KL reg: 0.001
Score: 0.797, KL reg: 0.001
Score: 0.798, KL reg: 0.001
Score: 0.798, KL reg: 0.001
Score: 0.798, KL reg: 0.001
Score: 0.798, KL reg: 0.001


INFO:root:Saving results..
INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


Cell types: ['Excitatory L6', 'Excitatory L5', 'Inhibitory Sst', 'Inhibitory Vip', 'Excitatory L4', 'Inhibitory Pvalb', 'Inhibitory Other', 'Excitatory L2/3', 'Astro', 'Other', 'Endo', 'Olig', 'Smc', 'Neuron Other', 'Micro']

Cell composition matrix shape: (189, 15)
Column names: ['Excitatory L6', 'Excitatory L5', 'Inhibitory Sst', 'Inhibitory Vip', 'Excitatory L4', 'Inhibitory Pvalb', 'Inhibitory Other', 'Excitatory L2/3', 'Astro', 'Other', 'Endo', 'Olig', 'Smc', 'Neuron Other', 'Micro']

First few rows:
   Excitatory L6  Excitatory L5  Inhibitory Sst  Inhibitory Vip  \
0       0.001560       0.003168        0.003801        0.002544   
1       0.004848       0.005129        0.009820        0.008211   
2       0.003011       0.002613        0.001876        0.002698   
3       0.003867       0.000552        0.005040        0.004414   
4       0.014510       0.005711        0.013779        0.008262   

   Excitatory L4  Inhibitory Pvalb  Inhibitory Other  Excitatory L2/3  \
0       0.002

In [6]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import tangram as tg
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置绘图参数
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')
plt.rcParams['figure.figsize'] = (10, 8)

print("Tangram version:", tg.__version__)
print("Scanpy version:", sc.__version__)

sc_path = "/mnt/d/ST_Graduation_Project_data/DATA24/scRNA.h5ad"
st_path = "/mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad"

print(f"Loading SC data from: {sc_path}")
adata_sc = sc.read_h5ad(sc_path)
print(f"SC data shape: {adata_sc.shape}")
print(f"SC obs columns: {adata_sc.obs.columns.tolist()}")

print(f"\nLoading ST data from: {st_path}")
adata_st = sc.read_h5ad(st_path)
print(f"ST data shape: {adata_st.shape}")
print(f"ST obs columns: {adata_st.obs.columns.tolist()}")

# 检查是否有空间坐标
if 'spatial' in adata_st.obsm:
    print(f"Spatial coordinates shape: {adata_st.obsm['spatial'].shape}")
else:
    print("Warning: No spatial coordinates found in ST data")

# 检查并确定细胞类型列名
# 常见的列名：'celltype', 'cell_type', 'cluster', 'leiden', 'louvain'
celltype_col = None
for col in ['celltype', 'cell_type', 'cluster_label', 'cluster', 'leiden', 'louvain']:
    if col in adata_sc.obs.columns:
        celltype_col = col
        break

if celltype_col is None:
    print("Warning: No celltype column found. Available columns:", adata_sc.obs.columns.tolist())
    # 如果没有聚类，先进行聚类
    print("Performing clustering on SC data...")
    sc.pp.neighbors(adata_sc, n_neighbors=15)
    sc.tl.leiden(adata_sc, resolution=0.5)
    celltype_col = 'leiden'
else:
    print(f"Using celltype column: {celltype_col}")

print(f"\nCell types found: {adata_sc.obs[celltype_col].unique()}")
print(f"Cell type counts:\n{adata_sc.obs[celltype_col].value_counts()}")

# 找到共同基因
common_genes = np.intersect1d(adata_sc.var_names, adata_st.var_names)
print(f"\nNumber of common genes: {len(common_genes)}")
print(f"SC genes: {len(adata_sc.var_names)}, ST genes: {len(adata_st.var_names)}")

# 确保使用共同基因
adata_sc_sub = adata_sc[:, common_genes].copy()
adata_st_sub = adata_st[:, common_genes].copy()

print(f"After subsetting - SC: {adata_sc_sub.shape}, ST: {adata_st_sub.shape}")

# Tangram 推荐：选择高变基因作为训练特征
# 这里我们使用 ST 数据中的高变基因
print("\nSelecting highly variable genes from ST data...")
sc.pp.highly_variable_genes(adata_st_sub, n_top_genes=2000, flavor='seurat_v3')
n_hvg = adata_st_sub.var['highly_variable'].sum()
print(f"Number of highly variable genes: {n_hvg}")

tg.pp_adatas(adata_sc_sub, adata_st_sub, genes=None)  # 使用所有共同基因，或指定 genes=training_genes

# 执行映射
ad_map = tg.map_cells_to_space(
    adata_sc=adata_sc_sub,
    adata_sp=adata_st_sub,
    mode='clusters', 
    cluster_label=celltype_col,  # 细胞类型列名
    density_prior='rna_count_based',  # 先验：基于 RNA count
    num_epochs=500,  # 训练轮数，可根据需要调整
    device='cuda:0' if __import__('torch').cuda.is_available() else 'cpu',  # 使用 GPU（如果有）
    learning_rate=0.1,
    verbose=True
)

print("\nTangram mapping completed!")
print(f"Mapping matrix shape: {ad_map.shape}")  # [n_cells/clusters, n_spots]

Tangram version: 1.0.4
Scanpy version: 1.11.4
Loading SC data from: /mnt/d/ST_Graduation_Project_data/DATA24/scRNA.h5ad
SC data shape: (10000, 31489)
SC obs columns: ['celltype_final', 'cell_type']

Loading ST data from: /mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad
ST data shape: (1000, 27345)
ST obs columns: ['cell_counts']
Using celltype column: cell_type

Cell types found: ['Proximal Tubule', 'Principal Cells', 'Thick Ascending Limb', 'Distal Convoluted Tubule', 'Endothelial Cell', 'Mesangial Cells', 'Connecting Tubule', 'Intercalated Cells Type A', 'Intercalated Cells Type B', 'Leukocyte']
Categories (10, object): ['Connecting Tubule', 'Distal Convoluted Tubule', 'Endothelial Cell', 'Intercalated Cells Type A', ..., 'Mesangial Cells', 'Principal Cells', 'Proximal Tubule', 'Thick Ascending Limb']
Cell type counts:
cell_type
Proximal Tubule              3632
Principal Cells              2045
Thick Ascending Limb         1409
Intercalated Cells Type A    1020
Endothelial Cell

INFO:root:15597 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:15597 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 15597 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.225, KL reg: 0.198
Score: 0.292, KL reg: 0.000
Score: 0.293, KL reg: 0.000


KeyboardInterrupt: 

In [ ]:
# ad_ge = tg.project_genes(
#     adata_map=ad_map,          # 上一步生成的映射结果
#     adata_sc=adata_sc_sub,     # 单细胞数据源
#     cluster_label=celltype_col # 必须与 map_cells_to_space 中的 cluster_label 一致
# )

# print("\nGene projection completed!")
# df_expression = ad_ge.to_df()
# df_expression.columns = df_expression.columns.str.upper()

# # 保存为 CSV
# df_expression.to_csv("predicted_spatial_expression.csv")


Gene projection completed!
Reconstructed expression shape: (1000, 17641)


In [4]:
# 从映射矩阵提取细胞类型组成
# ad_map.X 是 [n_clusters, n_spots] 的矩阵，表示每个簇在每个 spot 的概率/比例
# ad_map.obs 中直接包含 cell_type 列

# 直接使用 ad_map.obs 中的 cell_type
celltype_names = ad_map.obs['cell_type'].tolist()
print(f"Cell types: {celltype_names}")

# 转置以获得 [n_spots, n_celltypes] 的组成矩阵
cell_composition_tangram = pd.DataFrame(
    ad_map.X.T,  # 转置
    index=adata_st_sub.obs_names,  # spot 名称
    columns=celltype_names  # celltype 名称
)

print(f"\nCell composition matrix shape: {cell_composition_tangram.shape}")
print(f"Column names: {cell_composition_tangram.columns.tolist()}")
print(f"\nFirst few rows:\n{cell_composition_tangram.head()}")

# 验证每行和（应该接近 1）
row_sums = cell_composition_tangram.sum(axis=1)
print(f"\nRow sums - min: {row_sums.min():.4f}, max: {row_sums.max():.4f}, mean: {row_sums.mean():.4f}")

# 保存结果
# 🔧 归一化 Tangram 结果
print("\n" + "="*60)
print("NORMALIZING TANGRAM RESULTS")
print("="*60)

# 检查当前的行和
row_sums = cell_composition_tangram.sum(axis=1)
print(f"\n❌ BEFORE normalization:")
print(f"   Row sums - min: {row_sums.min():.6f}, max: {row_sums.max():.6f}, mean: {row_sums.mean():.6f}")
print(f"   Example: first row sum = {row_sums.iloc[0]:.6f}")

# 归一化：每行除以该行的和
cell_composition_tangram = cell_composition_tangram.div(row_sums, axis=0)

# 验证归一化后
row_sums_after = cell_composition_tangram.sum(axis=1)
print(f"\n✅ AFTER normalization:")
print(f"   Row sums - min: {row_sums_after.min():.6f}, max: {row_sums_after.max():.6f}, mean: {row_sums_after.mean():.6f}")
print(f"   Example: first row sum = {row_sums_after.iloc[0]:.6f}")

print(f"\nFirst row values (now as percentages):")
print(cell_composition_tangram.head())

# 重新保存归一化后的结果
cell_composition_tangram.to_csv("STARmap_tangram_composition.csv")
print(f"\n💾 Saved normalized results!")
print("="*60)

Cell types: ['Excitatory L6', 'Excitatory L5', 'Inhibitory Sst', 'Inhibitory Vip', 'Excitatory L4', 'Inhibitory Pvalb', 'Inhibitory Other', 'Excitatory L2/3', 'Astro', 'Other', 'Endo', 'Olig', 'Smc', 'Neuron Other', 'Micro']


NameError: name 'adata_st_sub' is not defined